# Stanford Stages - XNAT notebook demo

Automated sleep stage scoring (and narcolepsy features) from a PSG `.edf` recording.

**Before you run anything, switch this notebook to the `Python 3.7 (stanford-stages)` kernel**
(Kernel -> Change Kernel...). The classifier's dependencies live in that environment; the
default Python 3 kernel will not be able to import them.

Everything the classifier needs - the application code, the scaling/noise assets, and the
large `ac/` LSTM models - is baked into this image under `/opt/stanford-stages`. You only
supply an EDF recording and a place to write the results.

## 1. Confirm the environment

The `stanford-stages.pth` file baked into the 3.7 environment puts the module directory on
`sys.path`, so `import run_stanford_stages` works with no path setup.

In [ ]:
import sys, tensorflow, run_stanford_stages

print('python    :', sys.version.split()[0])
print('tensorflow:', tensorflow.__version__)
print('loaded    :', run_stanford_stages.__file__)

## 2. Configure the run

Start from the bundled config (`stanford_stages.xnat.json`, which already points at the
baked-in models) and override just the input and output.

- Point `edf_filename` at your recording. In XNAT the mounted study data appears under your
  workspace - adjust the path below to match. The image ships a sample recording at
  `/opt/stanford-stages/data/input/CHP040.edf` so you can try it end to end.
- Make sure `channel_labels` match the channel names in YOUR EDF header (the defaults match
  the sample recording).
- Results are written to `output_path`; put it somewhere writable in your workspace.

In [ ]:
import json, os, tempfile
from pathlib import Path

with open('/opt/stanford-stages/stanford_stages.xnat.json') as fid:
    config = json.load(fid)

# XNAT mounts each user's workspace at /workspace/<username>/.
username = os.environ.get('JUPYTERHUB_USER') or Path.home().name

# --- edit the edf_filename line for your data ---
config['edf_filename'] = '/opt/stanford-stages/data/input/CHP040.edf'
config['output_path'] = f'/workspace/{username}/stanford_stages_output'

Path(config['output_path']).mkdir(parents=True, exist_ok=True)

# run_using_json_file() takes a path, so write the edited config to a temp file.
config_path = Path(tempfile.gettempdir()) / 'stanford_stages_run.json'
config_path.write_text(json.dumps(config, indent=2))
print('wrote', config_path)

## 3. Run the classifier

This scores the recording across all 16 LSTM models and can take several minutes on CPU.

In [ ]:
run_stanford_stages.run_using_json_file(str(config_path))

## 4. Inspect the results

Outputs are written directly to `output_path`: the hypnogram (`.hypnogram.txt` / `.sta`),
the hypnodensity (`.hypnodensity.h5` / `.txt`), the encoding, and a hypnodensity plot.

In [ ]:
from IPython.display import Image, display

out_dir = Path(config['output_path'])
for path in sorted(out_dir.iterdir()):
    print(path.name)

plots = sorted(out_dir.glob('*.png'))
if plots:
    display(Image(filename=str(plots[0])))